In [ ]:
import os
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

In [ ]:
import os, sys, subprocess, gc, json, re, glob, shutil
from datetime import datetime
from tqdm import tqdm

# -------- CLEANUP PREVIOUS RUNS ---------
# Clear /kaggle/working to avoid picking up old output files
for f in os.listdir("/kaggle/working"):
    if f.startswith("gen_") or f == "images":
        try:
            path = os.path.join("/kaggle/working", f)
            if os.path.isdir(path): shutil.rmtree(path)
            else: os.remove(path)
        except: pass

def find_dataset_file(slug, filename):
    paths = [
        f"/kaggle/input/{slug}/{filename}",
        f"/kaggle/input/datasets/leventecsibi/{slug}/{filename}",
    ]
    for p in paths:
        if os.path.exists(p): return p
    found = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if found:
        found = [f for f in found if not f.endswith(".zip")]
        if found: return found[0]
    return None

HF_TOKEN_PATH = find_dataset_file("imggenhub-hf-token", "hf_token.json")
with open(HF_TOKEN_PATH, "r") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
os.environ["HF_TOKEN"] = HF_TOKEN

# -------- SETTINGS ---------
MODEL_SOURCE = "huggingface"
MODEL_ID = "city96/FLUX.1-schnell-gguf"
MODEL_FILENAME = "flux1-schnell-Q4_0.gguf"
VAE_REPO_ID = "black-forest-labs/FLUX.1-schnell"
VAE_FILENAME = "ae.safetensors"
CLIP_L_REPO_ID = "comfyanonymous/flux_text_encoders"
CLIP_L_FILENAME = "clip_l.safetensors"
T5XXL_REPO_ID = "comfyanonymous/flux_text_encoders"
T5XXL_FILENAME = "t5xxl_fp8_e4m3fn.safetensors"

from huggingface_hub import hf_hub_download
DIFFUSION_MODEL = hf_hub_download(repo_id=MODEL_ID, filename=MODEL_FILENAME, token=HF_TOKEN)
VAE_MODEL = hf_hub_download(repo_id=VAE_REPO_ID, filename=VAE_FILENAME, token=HF_TOKEN)
CLIP_L_MODEL = hf_hub_download(repo_id=CLIP_L_REPO_ID, filename=CLIP_L_FILENAME, token=HF_TOKEN)
T5XXL_MODEL = hf_hub_download(repo_id=T5XXL_REPO_ID, filename=T5XXL_FILENAME, token=HF_TOKEN)

# -------- PARAMETERS (injected) ---------
PROMPTS = ["test"]
OUTPUT_DIR = "."
INDEX_OFFSET = 0
IMG_SIZE = (512, 512)
GUIDANCE = 1.0
STEPS = 4
SEED = 42
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
import os, sys, subprocess, gc, json, re
from datetime import datetime
from tqdm import tqdm
# Setup stable-diffusion.cpp executable
!mkdir -p /kaggle/working/stable-diffusion.cpp
!cp -r /kaggle/input/datasets/leventecsibi/sd-build-zip/kaggle/working/export/sd_build/* /kaggle/working/stable-diffusion.cpp/
!chmod +x /kaggle/working/stable-diffusion.cpp/build/bin/sd

sd_executable = "/kaggle/working/stable-diffusion.cpp/build/bin/sd"
if not os.path.isfile(sd_executable):
    raise FileNotFoundError("sd executable not found at: " + sd_executable)

print(f"SD executable ready: {sd_executable}")


In [ ]:
import re
def slugify(text):
    text = text.lower()
    text = re.sub(r"[^\w\s-]", "", text)
    return re.sub(r"[\s-]+", "_", text).strip("_")

print(f"Generating {len(PROMPTS)} images...")
for i, prompt in enumerate(tqdm(PROMPTS, desc="Generating")):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_prompt = slugify(prompt)[:30]
    prompt_idx = i + 1 + INDEX_OFFSET
    filename = f"gen_p{prompt_idx}_{safe_prompt}_{timestamp}.png"
    output_path = os.path.join(OUTPUT_DIR, filename)
    width, height = IMG_SIZE
    
    cmd = [
        "/kaggle/working/stable-diffusion.cpp/build/bin/sd",
        "--diffusion-model", DIFFUSION_MODEL,
        "--vae", VAE_MODEL,
        "--clip_l", CLIP_L_MODEL,
        "--t5xxl", T5XXL_MODEL,
        "--prompt", prompt,
        "--cfg-scale", str(GUIDANCE),
        "--sampling-method", "euler",
        "--steps", str(STEPS),
        "--width", str(width),
        "--height", str(height),
        "--seed", str(SEED),
        "--output", output_path,
        "--rng", "cuda",
        "-v"
    ]
    
    subprocess.run(cmd, capture_output=True, text=True)
    if os.path.exists(output_path):
        print(f"Saved: {filename}")
    else:
        print(f"Failed: {filename}")
    gc.collect()

print("COMPLETE")